# 学生氏化 Bootstrap 置信区间（CHFS 家庭收入）

> 对应代码：`MathStatsCode/code_in_notes/Chap.15/bootstrap_student.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**实验目的**：对 CHFS 家庭总收入均值的 t 统计量做 1000 次 Bootstrap，用学生氏化（标准化）统计量的分位数作为临界值，手工构造置信区间。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


载入 2017 年 CHFS 家庭数据，`drop if total_income<0` 删除总收入为负（不适用）的观测，`set seed` 固定随机种子保证结果可复现。


In [2]:
%%stata
clear all
use ../datasets/chfs2017_hh.dta
drop if total_income<0
set seed 250211



. clear all

. use ../datasets/chfs2017_hh.dta

. drop if total_income<0
(566 observations deleted)

. set seed 250211

. 


**compute theta_hat**

先用 `su total_income` 计算原始样本（即“bootstrap 世界”中的真值）的均值 `theta_hat` 与标准误 `theta_se`（$s/\sqrt{N}$）。


In [3]:
%%stata
su total_income
local theta_hat = r(mean)
local theta_se = r(sd)/sqrt(r(N))



. su total_income

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
total_income |     39,445    91065.53    192618.2          0    5000000

. local theta_hat = r(mean)

. local theta_se = r(sd)/sqrt(r(N))

. 


**bootstrap**

`bootstrap (sqrt(r(N))*(r(mean)-theta_hat)/r(sd)), reps(1000): su total_income`：每次有放回重抽样后重新执行 `su`，并收集表达式 $\sqrt{N}(\bar{x}^{*}-\bar{x})/s^{*}$——这正是以原始样本均值 $\bar{x}$ 为“真值”的**学生氏化（标准化）统计量** $t^{*}$。由于 $t^{*}$ 是（渐近）基准统计量，其 bootstrap 分布可以更精确地近似 $t$ 的抽样分布。


In [4]:
%%stata
bootstrap (sqrt(r(N))*(r(mean)-`theta_hat')/r(sd)), reps(1000): su total_income


(running summarize on estimation sample)

         excluded from the resampling because of missing values or other
         reasons. To exclude observations, press Break, save the data, drop
         any observations that are to be excluded, and rerun bootstrap.

Bootstrap replications (1,000): 

.

..

..

..

...

..

1,000 done



Bootstrap results                                       Number of obs = 39,445
                                                        Replications  =  1,000

      Command: summarize total_income
        _bs_1: sqrt(r(N))*(r(mean)-91065.52641821354)/r(sd)



------------------------------------------------------------------------------
             |   Observed   Bootstrap                         Normal-based
             | coefficient  std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
       _bs_1 |          0   1.023505     0.00   1.000    -2.006034    2.006034


------------------------------------------------------------------------------


**读取 bootstrap 分布的分位数（临界值）**

`e(ci_percentile)` 保存了 bootstrap 结果的百分位数区间，对本例而言即 $\{t_b^{*}\}$ 的 2.5% 与 97.5% 分位数——也就是学生氏化置信区间所需要的**临界值** `cv_l` 与 `cv_r`。


In [5]:
%%stata
matrix cip = e(ci_percentile)
local cv_l=cip["ll",1]
local cv_r=cip["ul",1]
di "Critical Values = [`cv_l'" _skip "`cv_r']"



. matrix cip = e(ci_percentile)

. local cv_l=cip["ll",1]

. local cv_r=cip["ul",1]

. di "Critical Values = [`cv_l'" _skip "`cv_r']"
Critical Values = [-2.026365041732788 1.925478339195251]

. 


**compute CI**

用临界值构造学生氏化置信区间 $[\hat{\theta}-t_{\alpha/2}^{*}\se(\hat{\theta}),\;\hat{\theta}-t_{1-\alpha/2}^{*}\se(\hat{\theta})]$。与直接使用 $\pm1.96$ 的正态区间不同，这里的临界值来自 bootstrap 分布本身，因而能够捕捉收入分布右偏所带来的有限样本非对称性。


In [6]:
%%stata
local CIl = `theta_hat'-`cv_r'*`theta_se'
local CIu = `theta_hat'-`cv_l'*`theta_se'
di "CI = [`CIl'" _skip "`CIu']"



. local CIl = `theta_hat'-`cv_r'*`theta_se'

. local CIu = `theta_hat'-`cv_l'*`theta_se'

. di "CI = [`CIl'" _skip "`CIu']"
CI = [89198.11546504879 93030.78159507703]

. 
